In [1]:
# przetwarzanie danych
import numpy as np
import pandas as pd

# przekształcanie – normalizacja danych
from sklearn.preprocessing import MinMaxScaler

# wizualizacja
import matplotlib.pyplot as plt

# sieci neuronowe
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import SimpleRNN
from keras.layers import LSTM
from keras.layers import GRU

# ewaluacja modelu
from sklearn.metrics import mean_squared_error

In [3]:
dataset = pd.read_csv('airline-passengers.csv')
dataset['Month'] = pd.to_datetime(dataset['Month'])
dataset.set_index(['Month'], inplace=True)
dataset

,Passengers
Month,
1949-01-01,112
1949-02-01,118
1949-03-01,132
1949-04-01,129
1949-05-01,121
...,...
1960-08-01,606
1960-09-01,508
1960-10-01,461


In [7]:
import math
from tensorflow.keras import Input

def create_sequences(data, look_back):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i+look_back, 0])
        y.append(data[i+look_back, 0])
    X = np.array(X)
    y = np.array(y)
    return np.reshape(X, (X.shape[0], 1, X.shape[1])), y


# Przygotowanie danych
def prepare_data(dataset, train_frac=0.7, val_frac=0.15):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(dataset)

    train_end = int(len(scaled) * train_frac)
    val_end = train_end + int(len(scaled) * val_frac)

    train_data = scaled[:train_end]
    val_data = scaled[train_end:val_end]
    test_data = scaled[val_end:]

    return train_data, val_data, test_data, scaler

# Grid search
def grid_search_lstm(train_data, val_data, look_backs, unit_range):
    best_rmse = float('inf')
    best_config = None
    best_model = None

    for look_back in look_backs:
        X_train, y_train = create_sequences(train_data, look_back)
        X_val, y_val = create_sequences(val_data, look_back)

        for units in unit_range:
            model = Sequential()
            model.add(Input(shape=(1, look_back)))
            model.add(LSTM(units))
            model.add(Dense(1))
            model.compile(loss='mean_squared_error', optimizer='adam')

            model.fit(X_train, y_train, epochs=10, batch_size=1, verbose=0)
            predictions = model.predict(X_val)
            rmse = math.sqrt(mean_squared_error(y_val, predictions))

            print(f"look_back={look_back}, units={units}, RMSE={rmse:.5f}")

            if rmse < best_rmse:
                best_rmse = rmse
                best_config = (look_back, units)
                best_model = model

    return best_model, best_config, best_rmse


def evaluate_on_test(best_model, test_data, look_back, scaler):
    X_test, y_test = create_sequences(test_data, look_back)
    predictions = best_model.predict(X_test)
    predictions = scaler.inverse_transform(predictions.reshape(-1, 1))
    y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

    rmse = math.sqrt(mean_squared_error(y_test, predictions))
    print(f"Final Test RMSE: {rmse:.5f}")

    plt.figure(figsize=(10, 4))
    plt.plot(y_test, label='True')
    plt.plot(predictions, label='Prediction')
    plt.title("Best Model: Forecast vs True Values (Test Set)")
    plt.legend()
    plt.show()

    return rmse

In [8]:
train_data, val_data, test_data, scaler = prepare_data(dataset)

In [ ]:
# Wykonanie grid search
look_back_range = range(1, 13)
units_range = range(1, 13)
best_model, best_config, best_val_rmse = grid_search_lstm(train_data, val_data, look_back_range, units_range)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=1, units=1, RMSE=0.35896
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
look_back=1, units=2, RMSE=0.31207
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=1, units=3, RMSE=0.18617
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
look_back=1, units=4, RMSE=0.17069
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
look_back=1, units=5, RMSE=0.14335
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
look_back=1, units=6, RMSE=0.17324
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
look_back=1, units=7, RMSE=0.20064
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
look_back=1, units=8, RMSE=0.11212
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=1, units=9, RMSE=0.15120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=1, units=10, RMSE=0.14917
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=1, units=11, RMSE=0.11001
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=1, units=12, RMSE=0.09602
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=2, units=1, RMSE=0.16636
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=4, RMSE=0.12486
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=6, units=5, RMSE=0.12552
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=6, RMSE=0.12772
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=7, RMSE=0.12765
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=8, RMSE=0.12207
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=6, units=9, RMSE=0.11772
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=6, units=10, RMSE=0.11972
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=11, RMSE=0.12872
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=6, units=12, RMSE=0.13994
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=7, units=1, RMSE=0.16535
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=7, units=2, RMSE=0.12012
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=7, units=3, RMSE=0.13331
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
look_back=7, units=4, RMSE=0.14799
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
look_back=